In [1]:
import logging

import pandas as pd
import torch
import torchtext

from models import *
from preprocessing import *

print("PyTorch Version : {}".format(torch.__version__))
print("Torch Text Version : {}".format(torchtext.__version__))

/home/sigma/projects/PycharmProjects/toxic-comment-detection/venv/lib/python3.11/site-packages/torchtext/data/utils.py:105: UserWarning: Spacy model "en" could not be loaded, trying "en_core_web_sm" instead
  warnings.warn(


PyTorch Version : 2.0.1+cu118
Torch Text Version : 0.15.2+cpu


In [2]:
%env WANDB_SILENT=True


logger = logging.getLogger("wandb")
logger.setLevel(logging.ERROR)

env: WANDB_SILENT=True


In [3]:
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda:0")

device

device(type='cuda', index=0)

In [4]:
# Load Datasets And Create Data Loaders

train_df = pd.read_csv("data/train.csv")
train_df.describe()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
count,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000
mean,0.095844,0.009996,0.052948,0.002996,0.049364,0.008805
std,0.294379,0.099477,0.223931,0.054650,0.216627,0.093420
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [5]:
BATCH_SIZE = 256
# df_frac = 0.3
df_frac = 1.0
num_epochs = 50

## CLSTM (50 tokens - GlovE 300 - adam)

In [6]:
train_dl_50t_glove_300, val_dl_50t_glove_300 = get_dataloaders(
    target_convertor="glove_300",
    maximum_tokens=50,
    batch_size=BATCH_SIZE,
    dataset_fraction=df_frac,
)

/home/sigma/projects/PycharmProjects/toxic-comment-detection/venv/lib/python3.11/site-packages/torchtext/data/utils.py:105: UserWarning: Spacy model "en" could not be loaded, trying "en_core_web_sm" instead
  warnings.warn(
127656it [01:13, 1734.94it/s]
/home/sigma/projects/PycharmProjects/toxic-comment-detection/venv/lib/python3.11/site-packages/torchtext/data/utils.py:105: UserWarning: Spacy model "en" could not be loaded, trying "en_core_web_sm" instead
  warnings.warn(
31915it [00:19, 1677.32it/s]


In [7]:
clstm = CLSTM(
    embed_dim=300,
    conv_config={"num_channels": 50, "kernel_sizes": [1, 2, 3, 5]},
    optimizer_type="adam",
    learning_rate=0.001,
    maximum_tokens=50,
).to(device)

clstm.train_model(
    train_dataloader=train_dl_50t_glove_300,
    validation_dataloader=val_dl_50t_glove_300,
    num_of_epochs=num_epochs,
    device=device,
    name="CLSTM (50 tokens - Glove_300 - adam)",
    architecture="CLSTM",
)

Epoch 1:
{   'Training AUROC': 0.7321972250938416,
    'Training F1_score': 0.004439868964254856,
    'Training accuracy': 0.9595997333526611,
    'Training loss': 0.15132246063443605,
    'Training precision': 0.022409873083233833,
    'Training recall': 0.0024640217889100313,
    'Validation AUROC': 0.7735519409179688,
    'Validation F1_score': 0.0,
    'Validation accuracy': 0.9629484415054321,
    'Validation loss': 0.14198144263029097,
    'Validation precision': 0.0,
    'Validation recall': 0.0}
Epoch 2:
{   'Training AUROC': 0.8595337271690369,
    'Training F1_score': 0.2784758508205414,
    'Training accuracy': 0.9676879048347473,
    'Training loss': 0.11498291643788079,
    'Training precision': 0.7583360075950623,
    'Training recall': 0.17055314779281616,
    'Validation AUROC': 0.9774892330169678,
    'Validation F1_score': 0.7057596445083618,
    'Validation accuracy': 0.9793775081634521,
    'Validation loss': 0.05810180994868278,
    'Validation precision': 0.748656

## CLSTM (25 tokens - GlovE 300 - adam)

In [6]:
train_dl_25t_glove_300, val_dl_25t_glove_300 = get_dataloaders(
    target_convertor="glove_300",
    maximum_tokens=25,
    batch_size=BATCH_SIZE,
    dataset_fraction=df_frac,
)

127656it [01:07, 1893.49it/s]
31915it [00:14, 2143.81it/s]


In [7]:
clstm = CLSTM(
    embed_dim=300,
    conv_config={"num_channels": 50, "kernel_sizes": [1, 2, 3, 5]},
    optimizer_type="adam",
    learning_rate=0.001,
    maximum_tokens=25,
).to(device)

clstm.train_model(
    train_dataloader=train_dl_25t_glove_300,
    validation_dataloader=val_dl_25t_glove_300,
    num_of_epochs=num_epochs,
    device=device,
    name="CLSTM (25 tokens - Glove_300 - adam)",
    architecture="CLSTM",
)

Epoch 1:
{   'Training AUROC': 0.763803243637085,
    'Training F1_score': 0.00800979696214199,
    'Training accuracy': 0.960870087146759,
    'Training loss': 0.14692730450618244,
    'Training precision': 0.05908203125,
    'Training recall': 0.004296112339943647,
    'Validation AUROC': 0.8989577889442444,
    'Validation F1_score': 0.0,
    'Validation accuracy': 0.963794469833374,
    'Validation loss': 0.1080123256444931,
    'Validation precision': 0.0,
    'Validation recall': 0.0}
Epoch 2:
{   'Training AUROC': 0.9628047943115234,
    'Training F1_score': 0.6415549516677856,
    'Training accuracy': 0.9778976440429688,
    'Training loss': 0.06695460539333567,
    'Training precision': 0.7946915626525879,
    'Training recall': 0.5379016399383545,
    'Validation AUROC': 0.9723060131072998,
    'Validation F1_score': 0.6755146980285645,
    'Validation accuracy': 0.9800825119018555,
    'Validation loss': 0.05972891725599766,
    'Validation precision': 0.8234806060791016,
  

## CLSTM (50 tokens - GlovE 100 - adam)

In [6]:
train_dl_50t_glove_100, val_dl_50t_glove_100 = get_dataloaders(
    target_convertor="glove_100",
    maximum_tokens=50,
    batch_size=BATCH_SIZE,
    dataset_fraction=df_frac,
)

127656it [01:12, 1766.61it/s]
31915it [00:15, 2012.46it/s]


In [7]:
clstm = CLSTM(
    embed_dim=100,
    conv_config={"num_channels": 50, "kernel_sizes": [1, 2, 3, 5]},
    optimizer_type="adam",
    learning_rate=0.001,
    maximum_tokens=50,
).to(device)

clstm.train_model(
    train_dataloader=train_dl_50t_glove_100,
    validation_dataloader=val_dl_50t_glove_100,
    num_of_epochs=num_epochs,
    device=device,
    name="CLSTM (50 tokens - Glove_100 - adam)",
    architecture="CLSTM",
)

Epoch 1:
{   'Training AUROC': 0.735378086566925,
    'Training F1_score': 0.005445378366857767,
    'Training accuracy': 0.9613701701164246,
    'Training loss': 0.1499021589039323,
    'Training precision': 0.04392624646425247,
    'Training recall': 0.0029026016127318144,
    'Validation AUROC': 0.7885462045669556,
    'Validation F1_score': 0.0,
    'Validation accuracy': 0.9624419212341309,
    'Validation loss': 0.1441034430861473,
    'Validation precision': 0.0,
    'Validation recall': 0.0}
Epoch 2:
{   'Training AUROC': 0.9029680490493774,
    'Training F1_score': 0.34458959102630615,
    'Training accuracy': 0.9687559008598328,
    'Training loss': 0.10225088277715959,
    'Training precision': 0.7309166789054871,
    'Training recall': 0.22543539106845856,
    'Validation AUROC': 0.9573625326156616,
    'Validation F1_score': 0.5521242618560791,
    'Validation accuracy': 0.9737949967384338,
    'Validation loss': 0.07573904350399971,
    'Validation precision': 0.770937204

## CLSTM (25 tokens - GlovE 100 - adam)

In [6]:
train_dl_25t_glove_100, val_dl_25t_glove_100 = get_dataloaders(
    target_convertor="glove_100",
    maximum_tokens=25,
    batch_size=BATCH_SIZE,
    dataset_fraction=df_frac,
)

127656it [01:05, 1953.98it/s]
31915it [00:14, 2197.36it/s]


In [7]:
clstm = CLSTM(
    embed_dim=100,
    conv_config={"num_channels": 50, "kernel_sizes": [1, 2, 3, 5]},
    optimizer_type="adam",
    learning_rate=0.001,
    maximum_tokens=25,
).to(device)

clstm.train_model(
    train_dataloader=train_dl_25t_glove_100,
    validation_dataloader=val_dl_25t_glove_100,
    num_of_epochs=num_epochs,
    device=device,
    name="CLSTM (25 tokens - Glove_100 - adam)",
    architecture="CLSTM",
)

Epoch 1:
{   'Training AUROC': 0.7357032299041748,
    'Training F1_score': 0.006923549808561802,
    'Training accuracy': 0.9599248766899109,
    'Training loss': 0.15127847927486252,
    'Training precision': 0.041392650455236435,
    'Training recall': 0.0037777149118483067,
    'Validation AUROC': 0.7636854648590088,
    'Validation F1_score': 0.0,
    'Validation accuracy': 0.9646247625350952,
    'Validation loss': 0.13727742040157317,
    'Validation precision': 0.0,
    'Validation recall': 0.0}
Epoch 2:
{   'Training AUROC': 0.8719602227210999,
    'Training F1_score': 0.2777368426322937,
    'Training accuracy': 0.9667335152626038,
    'Training loss': 0.11371097020162849,
    'Training precision': 0.704486608505249,
    'Training recall': 0.17296285927295685,
    'Validation AUROC': 0.9477663040161133,
    'Validation F1_score': 0.5804213881492615,
    'Validation accuracy': 0.9747297763824463,
    'Validation loss': 0.07950257328152656,
    'Validation precision': 0.7032989